In [ ]:
SPEC={'embedding_plan_manifest_sha256': '3da94b191f6b56310d45c9e3cd7ccaf9c47b111d8eb77fc6571c890b8e556eb4'}
SOURCE='"""Offline Kaggle GPU inference for a verified incremental Qwen8B/Octen8B inventory."""\nfrom pathlib import Path\nfrom collections import Counter\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport shutil\nimport time\n\nimport numpy as np\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nPART_ROWS = 25_000\nDIM = 4096\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for data in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            h.update(data)\n    return h.hexdigest()\n\n\ndef atomic_json(path, data):\n    temp = path.with_suffix(\'.json.tmp\')\n    temp.write_text(json.dumps(data, indent=2), encoding=\'utf-8\')\n    os.replace(temp, path)\n\n\ndef vector_stats(vectors):\n    assert vectors.ndim == 2 and vectors.shape[1] == DIM\n    assert vectors.dtype == np.float16 and np.isfinite(vectors).all()\n    norms = np.linalg.norm(vectors.astype(np.float32), axis=1)\n    assert not len(norms) or np.max(np.abs(norms - 1)) < 0.01\n    return dict(nan_rows=0, norm_min=float(norms.min()) if len(norms) else None,\n                norm_max=float(norms.max()) if len(norms) else None)\n\n\ndef fill_vectors(rows, embed, old_vectors, lookup):\n    """All inference/copy dependencies are explicit; no uninitialized canonical rows."""\n    result = np.empty((len(rows), DIM), dtype=np.float16)\n    ready = np.zeros(len(rows), dtype=bool)\n    start = rows[0][\'output_row\'] if rows else 0\n    assert [r[\'output_row\'] for r in rows] == list(range(start, start + len(rows)))\n    infer = [i for i, r in enumerate(rows) if r[\'action\'] == \'infer\']\n    for offset in range(0, len(infer), 4096):\n        indices = infer[offset:offset + 4096]\n        vectors = embed([rows[i][\'chunk_text\'] for i in indices])\n        assert vectors.shape == (len(indices), DIM)\n        vector_stats(vectors)\n        result[indices] = vectors\n        ready[indices] = True\n    for i, row in enumerate(rows):\n        if row[\'action\'] == \'copy_old\':\n            pos = row[\'old_reuse_row\']\n            assert 0 <= pos < len(old_vectors)\n            result[i] = old_vectors[pos]\n            ready[i] = True\n        elif row[\'action\'] == \'copy_new\':\n            prior = row[\'canonical_output_row\']\n            assert 0 <= prior < row[\'output_row\']\n            if prior >= start:\n                assert ready[prior - start]\n                result[i] = result[prior - start]\n            else:\n                result[i] = lookup(prior)\n            ready[i] = True\n        else:\n            assert row[\'action\'] == \'infer\'\n    assert ready.all()\n    vector_stats(result)\n    return result\n\n\ndef make_embedder(model_path, contract):\n    import torch\n    import torch.nn.functional as F\n    from transformers import AutoModel, AutoTokenizer\n\n    assert torch.cuda.is_available()\n    gpu = torch.cuda.get_device_name(0)\n    assert \'6000\' in gpu, gpu\n    torch.set_num_threads(2)\n    tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)\n    tokenizer.padding_side = \'left\'\n    started = time.monotonic()\n    print(\'Loading pinned\', contract[\'model\'], \'on\', gpu, flush=True)\n    model = AutoModel.from_pretrained(str(model_path), torch_dtype=torch.bfloat16,\n        attn_implementation=\'sdpa\', local_files_only=True).cuda().eval()\n    assert model.config.hidden_size == DIM\n    print(f\'Model ready after {time.monotonic() - started:.1f}s\', flush=True)\n    tally = dict(forward_inputs=0, tokens=0, inputs_at_length_cap=0)\n\n    @torch.inference_mode()\n    def embed(texts):\n        ids = tokenizer([contract[\'doc_prefix\'] + t for t in texts], truncation=True, max_length=1024)[\'input_ids\']\n        order = np.argsort([-len(x) for x in ids], kind=\'stable\')\n        output = np.empty((len(texts), DIM), dtype=np.float16)\n        for offset in range(0, len(order), 16):\n            index = order[offset:offset + 16]\n            enc = tokenizer.pad({\'input_ids\': [ids[i] for i in index]}, return_tensors=\'pt\').to(\'cuda\')\n            hidden = model(**enc, use_cache=False).last_hidden_state\n            mask = enc[\'attention_mask\']\n            if mask[:, -1].sum() == mask.shape[0]:\n                pooled = hidden[:, -1]\n            else:\n                lengths = mask.sum(dim=1) - 1\n                pooled = hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]\n            output[index] = F.normalize(pooled.float(), p=2, dim=1).cpu().numpy().astype(np.float16)\n            tally[\'forward_inputs\'] += len(index)\n            tally[\'tokens\'] += sum(len(ids[i]) for i in index)\n            tally[\'inputs_at_length_cap\'] += sum(len(ids[i]) == 1024 for i in index)\n            if offset % 800 == 0:\n                print(f"Forward inputs incl. compatibility checks: {tally[\'forward_inputs\']:,}", flush=True)\n        return output\n    return embed, tally, dict(gpu=gpu, torch=torch.__version__, transformers=importlib.metadata.version(\'transformers\'))\n\n\ndef run(spec, input_root=\'/kaggle/input\', output_root=\'/kaggle/working\'):\n    started = time.monotonic()\n    os.environ.update(HF_HUB_OFFLINE=\'1\', TRANSFORMERS_OFFLINE=\'1\', TOKENIZERS_PARALLELISM=\'false\')\n    root = Path(input_root)\n    plans = list(root.rglob(\'embedding_plan_manifest.json\'))\n    assert len(plans) == 1, plans\n    plan_path = plans[0]\n    assert sha(plan_path) == spec[\'embedding_plan_manifest_sha256\']\n    plan = json.loads(plan_path.read_text())\n    assert plan[\'dim\'] == DIM and {\'Qwen/Qwen3-Embedding-8B\': \'\', \'Octen/Octen-Embedding-8B\': \' \'}[plan[\'model\']] == plan[\'doc_prefix\']\n    assert plan[\'max_length\'] == 1024 and plan[\'batch\'] == 16\n    assert plan[\'inference_dtype\'] == \'torch.bfloat16\' and plan[\'output_dtype\'] == \'float16\'\n    assert plan[\'instruction\'] == \'\' and plan[\'pooling\'] == \'last_token\'\n    assert plan[\'normalize\'] and plan[\'padding_side\'] == \'left\'\n    for name, meta in plan[\'files\'].items():\n        path = plan_path.parent / name\n        assert path.stat().st_size == meta[\'bytes\'] and sha(path) == meta[\'sha256\'], name\n    packages = list(root.rglob(\'MODELS.json\'))\n    assert len(packages) == 1, packages\n    package_path = packages[0]\n    assert sha(package_path) == plan[\'model_package_manifest_sha256\']\n    entry = json.loads(package_path.read_text())[\'models\'][plan[\'model\']]\n    assert entry[\'sha\'] == plan[\'model_revision\']\n    model_path = package_path.parent / Path(entry[\'folder\']).name\n    for item in entry[\'files\']:\n        path = model_path / item[\'path\']\n        assert path.is_file() and path.stat().st_size == item[\'bytes\'], path\n    rows = pq.read_table(plan_path.parent / \'embedding_tasks.parquet\').to_pylist()\n    assert len(rows) == plan[\'output_chunks\']\n    assert [r[\'output_row\'] for r in rows] == list(range(len(rows)))\n    assert len({r[\'chunk_id\'] for r in rows}) == len(rows)\n    assert dict(Counter(r[\'action\'] for r in rows)) == plan[\'actions\']\n    for row in rows:\n        assert hashlib.sha256((plan[\'doc_prefix\']+row[\'chunk_text\']).encode()).hexdigest() == row[\'input_sha256\']\n    old_vectors = np.load(plan_path.parent / \'reused_old_vectors.npy\', mmap_mode=\'r\', allow_pickle=False)\n    assert old_vectors.shape == (plan[\'old_reuse_unique_inputs\'], DIM)\n    vector_stats(old_vectors)\n    out = Path(output_root)\n    out.mkdir(parents=True, exist_ok=True)\n    (out / \'vectors\').mkdir(exist_ok=True)\n    manifest_path = out / \'incremental_embed_manifest.json\'\n    manifest = dict(model=plan[\'model\'], model_revision=plan[\'model_revision\'], dim=DIM,\n        dtype=\'torch.bfloat16\', output_dtype=\'float16\', batch=16, max_length=1024, doc_prefix=plan[\'doc_prefix\'],\n        instruction=\'\', pooling=\'last_token\', padding_side=\'left\', normalize=True,\n        source_plan_sha256=sha(plan_path), source_release_manifest_sha256=plan[\'source_release_manifest_sha256\'],\n        model_package_manifest_sha256=plan[\'model_package_manifest_sha256\'],\n        job_code_sha256=sha(Path(__file__)), total_rows=len(rows), parts_total=(len(rows)+PART_ROWS-1)//PART_ROWS,\n        part_rows=PART_ROWS, actions=plan[\'actions\'], already_embedded_chunk_ids=plan[\'already_embedded_chunk_ids\'],\n        baseline_chunks_verified=plan[\'old_chunks_verified\'], baseline_modified=False,\n        full_baseline_reembedding=False, complete=False, parts=[], completed_rows=0,\n        parts_done=0, nan_rows=0, resumed_rows=0)\n    # A later run may mount an earlier interrupted job. Only verified sealed parts are reused.\n    candidates=[]\n    for candidate in root.rglob(\'incremental_embed_manifest.json\'):\n        previous=json.loads(candidate.read_text())\n        if all(previous.get(k)==manifest[k] for k in (\'source_plan_sha256\',\'model_revision\',\'job_code_sha256\',\'part_rows\')):\n            candidates.append((len(previous[\'parts\']),candidate,previous))\n    if candidates:\n        _, candidate, previous=max(candidates,key=lambda item:item[0])\n        for index, part in enumerate(previous[\'parts\']):\n            assert part[\'part\']==index and part[\'start_row\']==manifest[\'completed_rows\']\n            expected=rows[part[\'start_row\']:part[\'start_row\']+part[\'rows\']]\n            assert part[\'rows\']==min(PART_ROWS,len(rows)-part[\'start_row\'])\n            for key in (\'vectors\',\'ids\'):\n                item=part[key];source=candidate.parent/item[\'path\'];target=out/item[\'path\']\n                assert source.stat().st_size==item[\'bytes\'] and sha(source)==item[\'sha256\']\n                shutil.copyfile(source,target)\n            ids=pq.read_table(out/part[\'ids\'][\'path\'],columns=[\'chunk_id\']).column(0).to_pylist()\n            assert ids==[r[\'chunk_id\'] for r in expected]\n            v=np.load(out/part[\'vectors\'][\'path\'],mmap_mode=\'r\',allow_pickle=False)\n            assert v.shape==(len(expected),DIM);vector_stats(v);del v\n            manifest[\'parts\'].append(part);manifest[\'completed_rows\']+=part[\'rows\']\n        manifest[\'resumed_rows\']=manifest[\'completed_rows\'];manifest[\'parts_done\']=len(manifest[\'parts\'])\n    atomic_json(manifest_path,manifest)\n    embed,tally,hardware=make_embedder(model_path,plan)\n    manifest.update(hardware)\n    gold=pq.read_table(plan_path.parent/\'golden_texts.parquet\').to_pylist()\n    expected=np.load(plan_path.parent/\'golden_vectors.npy\',allow_pickle=False).astype(np.float32)\n    actual=embed([r[\'chunk_text\'] for r in gold]).astype(np.float32)\n    cosine=np.sum(actual*expected,axis=1)/(np.linalg.norm(actual,axis=1)*np.linalg.norm(expected,axis=1))\n    manifest[\'compatibility_check_old_texts\']=len(gold)\n    manifest[\'compatibility_cosine_min\']=float(cosine.min())\n    manifest[\'compatibility_cosine_mean\']=float(cosine.mean())\n    assert len(gold)==plan[\'golden_checks\'] and np.isfinite(cosine).all() and cosine.min()>=0.999, cosine\n    print(f\'PASS: {len(gold)} baseline compatibility checks; minimum cosine {cosine.min():.7f}\',flush=True)\n    atomic_json(manifest_path,manifest)\n    cache={}\n    def lookup(output_row):\n        part=manifest[\'parts\'][output_row//PART_ROWS]\n        key=part[\'vectors\'][\'path\']\n        if key not in cache:\n            cache.clear()\n            cache[key]=np.load(out/key,mmap_mode=\'r\',allow_pickle=False)\n        return cache[key][output_row-part[\'start_row\']]\n    for index in range(len(manifest[\'parts\']),manifest[\'parts_total\']):\n        start=index*PART_ROWS;current=rows[start:start+PART_ROWS]\n        part_started=time.monotonic()\n        vectors=fill_vectors(current,embed,old_vectors,lookup)\n        stem=f\'vectors/additions-20261029-v2-part-{index:03d}\'\n        vp=out/(stem+\'.npy\');ip=out/(stem+\'-ids.parquet\')\n        temp=vp.with_suffix(\'.npy.tmp\')\n        with temp.open(\'wb\') as handle:np.save(handle,vectors,allow_pickle=False)\n        os.replace(temp,vp)\n        ids=pa.Table.from_pylist([{k:r[k] for k in (\'chunk_id\',\'input_sha256\',\'lang\',\'source_chunk_row\')} for r in current])\n        temp=ip.with_suffix(\'.parquet.tmp\');pq.write_table(ids,temp,compression=\'zstd\');os.replace(temp,ip)\n        assert pq.ParquetFile(ip).metadata.num_rows==len(current)\n        part=dict(part=index,start_row=start,rows=len(current),actions=dict(Counter(r[\'action\'] for r in current)),\n            elapsed_seconds=round(time.monotonic()-part_started,1),**vector_stats(vectors))\n        for key,path in [(\'vectors\',vp),(\'ids\',ip)]:\n            part[key]=dict(path=path.relative_to(out).as_posix(),bytes=path.stat().st_size,sha256=sha(path))\n        manifest[\'parts\'].append(part);manifest[\'parts_done\']=index+1\n        manifest[\'completed_rows\']+=len(current)\n        manifest[\'elapsed_seconds\']=round(time.monotonic()-started,1)\n        manifest[\'forward_tally_including_compatibility\']=dict(tally)\n        atomic_json(manifest_path,manifest)\n        print(f\'SEALED part {index}: {len(current):,} rows; {manifest["completed_rows"]:,}/{len(rows):,}\',flush=True)\n        del vectors\n    assert manifest[\'completed_rows\']==len(rows)\n    assert Counter({k:sum(p[\'actions\'].get(k,0) for p in manifest[\'parts\']) for k in plan[\'actions\']})==Counter(plan[\'actions\'])\n    manifest[\'complete\']=True\n    manifest[\'elapsed_seconds\']=round(time.monotonic()-started,1)\n    atomic_json(manifest_path,manifest)\n    print(\'COMPLETE\',json.dumps({k:v for k,v in manifest.items() if k!=\'parts\'}),flush=True)\n\n\nif __name__==\'__main__\':\n    run(json.loads(Path(\'/kaggle/working/incremental_embed_spec.json\').read_text()))\n'
from pathlib import Path
import json, subprocess, sys
p=Path('/kaggle/working/embed_8b_delta.py'); p.write_text(SOURCE, encoding='utf-8')
Path('/kaggle/working/incremental_embed_spec.json').write_text(json.dumps(SPEC))
subprocess.run([sys.executable, '-u', str(p)], check=True)
